In [ ]:
# V0 -- load both pilot result files and pair them by (method, sample_id). Read-only.
# KB v3 SummaC is taken from the v2-premise columns (M1b), so both pilots are judged against the same
# KB v2 definitions; summac_src_def_v3p = v3 against its own lay-definition premise, kept for reference.
# Depends on: nothing. Peak RAM: < 50 MB.
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

project_dir = Path(r"D:\Project\Radiology_RAG")
V2 = project_dir/"results"/"pilot_kb_v2_matched"/"val_kb_v2m_qwen25_1.5b_scored.csv"
V3 = project_dir/"results"/"pilot_kb_v3"/"val_kb_v3_qwen25_1.5b_scored_v2premise.csv"

v2 = pd.read_csv(V2)
v3 = pd.read_csv(V3).rename(columns={"summac_src_def": "summac_src_def_v3p",
                                     "summac_src_def_worst": "summac_src_def_worst_v3p",
                                     "summac_src_def_v2p": "summac_src_def",
                                     "summac_src_def_worst_v2p": "summac_src_def_worst"})
assert len(v2) == len(v3) == 600, (len(v2), len(v3))
p = v2.merge(v3, on=["model", "method", "sample_id"], suffixes=("_v2", "_v3"), validate="one_to_one")
assert len(p) == 600

# sanity: no_rag uses no KB, so its answers must be identical in both pilots
nr = p[p.method == "no_rag"]
print(f"no_rag answers identical in both pilots: {(nr.answer_v2 == nr.answer_v3).sum()}/{len(nr)}")
print("methods:", sorted(p.method.unique()))

In [ ]:
# V1 -- KB v3 minus matched KB v2, per method, paired over the same 100 captions.
# Wilcoxon signed-rank (two-sided), Holm-corrected over the 5 retrieval methods within each metric.
# no_rag is left out: it does not use the KB (V0 shows it is identical).
# 100 captions is a pilot: read differences as direction + size, not as final evidence.
# Depends on: V0. Peak RAM: ~0 MB.
METRICS = ["summac_src_def", "summac_src_def_worst", "summac_ref",
           "term_kept", "term_glossed", "term_bare", "fkgl", "fre",
           "gold_cui_recall", "n_docs", "context_words", "answer_words"]
METHODS = ["oracle", "hybrid", "hybrid_anchored", "1hop", "2hop"]

def holm(ps):
    ps = np.asarray(ps, float); order = np.argsort(ps); adj = np.empty_like(ps); run = 0
    for k, i in enumerate(order):
        run = max(run, (len(ps) - k) * ps[i]); adj[i] = min(run, 1.0)
    return adj

rows = []
for m in METRICS:
    block = []
    for meth in METHODS:
        g = p[p.method == meth][[f"{m}_v2", f"{m}_v3"]].dropna()
        diff = g[f"{m}_v3"] - g[f"{m}_v2"]
        try:
            pv = wilcoxon(g[f"{m}_v3"], g[f"{m}_v2"]).pvalue if (diff != 0).any() else 1.0
        except ValueError:
            pv = 1.0
        block.append({"metric": m, "method": meth, "n": len(g),
                      "v2": g[f"{m}_v2"].mean(), "v3": g[f"{m}_v3"].mean(), "diff": diff.mean(),
                      "v3_better_%": 100 * (diff > 0).mean(), "v3_worse_%": 100 * (diff < 0).mean(), "p": pv})
    for r, a in zip(block, holm([r["p"] for r in block])):
        r["p_holm"] = a
    rows += block

res = pd.DataFrame(rows)
pd.set_option("display.width", 200)
for m in METRICS:
    print(f"\n=== {m}")
    print(res[res.metric == m].drop(columns="metric").round(3).to_string(index=False))

In [ ]:
# V2 -- link the v3-v2 change in explanation and faithfulness to the change in documents per prompt.
# Spearman correlation per answer, all 5 retrieval methods pooled (500 pairs).
# Depends on: V0. Peak RAM: ~0 MB.
from scipy.stats import spearmanr
q = p[p.method.isin(METHODS)].copy()
q["d_docs"]   = q.n_docs_v3 - q.n_docs_v2
q["d_recall"] = q.gold_cui_recall_v3 - q.gold_cui_recall_v2
for m in ["term_glossed", "term_bare", "summac_src_def", "fkgl"]:
    q[f"d_{m}"] = q[f"{m}_v3"] - q[f"{m}_v2"]
    for x in ["d_docs", "d_recall"]:
        s = q[[x, f"d_{m}"]].dropna()
        r, pv = spearmanr(s[x], s[f"d_{m}"])
        print(f"{x:9s} vs d_{m:15s} rho = {r:+.3f}  p = {pv:.2g}  n = {len(s)}")
print("\nanswers where v3 had fewer docs than v2:", f"{100*(q.d_docs < 0).mean():.1f}%",
      "| more:", f"{100*(q.d_docs > 0).mean():.1f}%")

In [ ]:
# V3 -- oracle answers where KB v3 glossed more terms than v2, and where it glossed fewer.
# Depends on: V0, V2. Peak RAM: ~0 MB.
o = q[q.method == "oracle"].dropna(subset=["d_term_glossed"])
for label, sub in [("v3 glossed MORE", o[o.d_term_glossed > 0]), ("v3 glossed FEWER", o[o.d_term_glossed < 0])]:
    print(f"\n########## {label}: {len(sub)} captions (showing up to 4)")
    for _, r in sub.head(4).iterrows():
        print(f"--- sample {r.sample_id} | glossed {r.term_glossed_v2:.2f} -> {r.term_glossed_v3:.2f}")
        print(f"   v2: {r.answer_v2}")
        print(f"   v3: {r.answer_v3}")